# Intro to Machine Learning — HW03: Gradient Descent & Ridge Regression

This notebook implements and validates the seven functions exercised by the official
autograder file `hw03_tests.py`:

| # | Function | Tested by | What it does |
|---|----------|-----------|--------------|
| 1 | `gd` | `TestGradientDescent` | Batch gradient descent with an analytic gradient |
| 2 | `num_grad` | `TestNumGrad` | Central-difference numerical gradient |
| 3 | `minimize` | `TestMinimize` | Gradient descent that needs **no** analytic gradient |
| 4 | `sgd` | `TestSGD` | Stochastic gradient descent (one random point per step) |
| 5 | `objective_func` | `TestObjectiveFunc` | Ridge regression objective $J(\theta)$ |
| 6 | `objective_func_grad` | `TestObjectiveFuncGrad` | Analytic gradient $\nabla_\theta J$ |
| 7 | `ridge_gd` | `TestRidgeGD` | Fits ridge regression by running `gd` on 5 & 6 |

**Structure of each section:** the maths → the contract (shapes in / shapes out) → the
implementation → the official test → a short interpretation of the result.

At the end there is a full regression run plus diagnostic plots (convergence, step-size
sensitivity, GD vs. the closed-form ridge solution).

---
## 0. Setup — download the test file with `wget`

`hw03_tests.py` is fetched straight from the course site so the notebook always validates
against the *current* reference values. The test module imports `scikit-learn`
(`KFold`, `StandardScaler`), which is pre-installed on Colab.

In [ ]:
import importlib
import numpy as np

import hw03_tests
importlib.reload(hw03_tests)          # picks up a freshly downloaded file
import hw03_tests as tests

# Helpers that the tests also use, re-exported for convenience.
from hw03_tests import rv, cv, lin_reg, mse, ridge_obj, ridge_analytic

np.set_printoptions(precision=6, suppress=False)
print("hw03_tests loaded. Public test entry points:")
print([n for n in dir(tests) if n.startswith("Test")])

In [ ]:
# Sanity check on the two column/row vector helpers used throughout.
print("rv([1,2,3]) shape:", rv([1, 2, 3]).shape)   # (1, 3)
print("cv([1,2,3]) shape:", cv([1, 2, 3]).shape)   # (3, 1)
print(cv([1.0, 2.0, 3.0]))

---
## 1. `gd` — batch gradient descent

### The idea
Gradient descent minimises $f(x)$ by repeatedly stepping *against* the gradient, because
$-\nabla f(x)$ is the direction of steepest local decrease:

$$x^{(t+1)} = x^{(t)} - \eta(t)\, \nabla f\!\left(x^{(t)}\right), \qquad t = 0, 1, \dots, T-1$$

$\eta(t)$ is supplied as a **function of the iteration index**, which lets you use a decaying
schedule such as $\eta(t)=1/\sqrt{t+1}$ instead of a constant.

### Contract
| Argument | Meaning |
|---|---|
| `f` | objective, takes `x` of shape `(d, 1)` → scalar or `(1, 1)` |
| `df` | gradient of `f`, takes `(d, 1)` → `(d, 1)` |
| `x0` | starting point, `(d, 1)` |
| `step_size_fn` | `int -> float`, called with the iteration index |
| `max_iter` | number of **updates** to perform |

**Returns** `(x, f(x))` — the final point and the objective value there.

### Two details the tests pin down
1. The loop performs **exactly `max_iter` updates**. In test 1, $f(x)=x_1^2+x_2$ has
   $\partial f/\partial x_2 = 1$, so $x_2$ moves $1 \to 1 - 0.1 \times 1000 = -99$.
   An off-by-one loop would return $-98.9$ and fail.
2. The returned objective is evaluated at the **final** `x`, not the last-visited one.

Note also test 2: $f(x)=x_0x_1$ is a saddle, not a bowl. Gradient descent happily diverges
to $\approx -1.1\times10^8$ — a useful reminder that GD only finds minima on functions that
have them, and only local ones at that.

In [ ]:
def gd(f, df, x0, step_size_fn, max_iter):
    """
    Batch gradient descent.

    f             : (d, 1) -> scalar   objective
    df            : (d, 1) -> (d, 1)   gradient of f
    x0            : (d, 1)             initial guess
    step_size_fn  : int -> float       learning-rate schedule, called with iteration index
    max_iter      : int                number of updates to perform

    Returns (x, f(x)) where x is (d, 1).
    """
    x = x0
    for t in range(max_iter):
        # x - eta*grad creates a NEW array, so the caller's x0 is never mutated.
        x = x - step_size_fn(t) * df(x)
    return x, f(x)

In [ ]:
tests.TestGradientDescent(gd)

A quick manual look at what the test just checked, so the numbers are not a black box.

In [ ]:
from hw03_tests import f1, df1, f2, df2

x_star, f_star = gd(f1, df1, cv([1.0, 1.0, 1.0]), lambda t: 0.1, 1000)
print("f1 = x1^2 + x2")
print("  x* =", x_star.ravel())
print("  x0 untouched (never used by f1), x1 -> 0.8^1000 =", 0.8 ** 1000)
print("  x2 -> 1 - 0.1*1000 =", 1 - 0.1 * 1000)
print("  f(x*) =", float(np.asarray(f_star).squeeze()))

x_star2, f_star2 = gd(f2, df2, cv([2.0, 3.0, 4.0]), lambda t: 0.01, 1000)
print("\nf2 = x0*x1 (a saddle - GD diverges)")
print("  x* =", x_star2.ravel())
print("  f(x*) =", float(np.asarray(f_star2).squeeze()))

---
## 2. `num_grad` — numerical gradient

### The idea
When the analytic gradient is unavailable (or you want to check one you derived by hand),
approximate each partial derivative with a **central difference**:

$$\frac{\partial f}{\partial x_i}(x) \;\approx\; \frac{f(x + \delta e_i) - f(x - \delta e_i)}{2\delta}$$

where $e_i$ is the $i$-th standard basis vector. Central differences have error
$O(\delta^2)$, versus $O(\delta)$ for the one-sided version $\frac{f(x+\delta)-f(x)}{\delta}$
— which is why this form is used.

$\delta = 0.001$ is the value the reference outputs are built on: the expected answer for
$\partial(x_1^2)/\partial x_1$ at $x_1=1$ is `2.0000000000000018`, exactly what
$\frac{1.001^2-0.999^2}{0.002}$ gives in float64.

### Contract
`num_grad(f)` is a **higher-order function**: it returns *another function* `df(x)` with the
same signature `(d, 1) -> (d, 1)` that `gd` expects. That is why the test calls it as
`num_grad(f)(x)`.

### Implementation notes
- Build a fresh perturbation vector each coordinate; never modify `x` in place, or later
  coordinates would be evaluated at a corrupted point.
- Each partial derivative costs **two** function evaluations, so a `d`-dimensional gradient
  costs `2d` — cheap here, prohibitive for real models. That cost is precisely the
  motivation for deriving analytic gradients (section 6) and for backpropagation.

In [ ]:
def num_grad(f, delta=0.001):
    """
    Central-difference numerical gradient.

    f     : (d, 1) -> scalar
    delta : perturbation size (0.001 matches the reference values in hw03_tests)

    Returns a function df: (d, 1) -> (d, 1).
    """
    def df(x):
        d = x.shape[0]
        g = np.zeros((d, 1))
        for i in range(d):
            e = np.zeros((d, 1))
            e[i, 0] = delta                      # perturb coordinate i only
            diff = (f(x + e) - f(x - e)) / (2 * delta)
            # f may hand back a float or a (1, 1) array; squeeze covers both.
            g[i, 0] = float(np.squeeze(diff))
        return g
    return df

In [ ]:
tests.TestNumGrad(num_grad)

**How accurate is it, and why not use a tiny `delta`?** Shrinking $\delta$ reduces
*truncation* error but amplifies *floating-point cancellation* — $f(x+\delta)$ and
$f(x-\delta)$ become nearly equal and their difference loses significant digits. The sweet
spot for central differences in float64 sits near $\delta \approx 10^{-5}$; below that the
error grows again.

In [ ]:
# f1 is quadratic, and central differences are EXACT on quadratics - a bad test bed.
# Use a genuinely non-polynomial function so the truncation error is visible.
def g_test(x):
    return np.exp(x[0:1, :]) * np.sin(x[1:2, :]) + x[2:3, :] ** 3

def dg_test(x):
    a, b, c = x.ravel()
    return cv([np.exp(a) * np.sin(b), np.exp(a) * np.cos(b), 3 * c ** 2])

x_test = cv([1.0, 2.0, 3.0])
truth = dg_test(x_test)

print(f"{'delta':>10} | {'max abs error':>15} | dominant error source")
print("-" * 58)
for delta in [1e-1, 1e-2, 1e-3, 1e-5, 1e-7, 1e-10, 1e-13]:
    err = np.max(np.abs(num_grad(g_test, delta)(x_test) - truth))
    source = "truncation O(delta^2)" if delta > 1e-5 else "float64 cancellation"
    print(f"{delta:>10.0e} | {err:>15.3e} | {source}")

---
## 3. `minimize` — gradient descent without an analytic gradient

Composition of the previous two pieces: build a numerical gradient for `f`, then hand both
to `gd`. Same return contract, `(x, f(x))`.

The interesting part is what the test reveals about **error accumulation**. With the exact
gradient, $x_1$ shrinks to $0.8^{1000} \approx 1.2\times10^{-97}$. With the numerical
gradient it stalls around $2.4\times10^{-14}$ — the difference quotient can't resolve a
gradient smaller than the float64 noise floor, so the updates stop shrinking $x_1$ once it
is tiny. Likewise $x_2$ lands on $-99.00000000010797$ rather than exactly $-99$: a
$\sim10^{-13}$ error per step, compounded over 1000 steps.

In [ ]:
def minimize(f, x0, step_size_fn, max_iter):
    """
    Gradient descent on f using a numerically estimated gradient.

    Returns (x, f(x)), matching gd's contract.
    """
    return gd(f, num_grad(f), x0, step_size_fn, max_iter)

In [ ]:
tests.TestMinimize(minimize)

In [ ]:
# Analytic vs numerical gradient, same problem and schedule.
x_exact, _ = gd(f1, df1, cv([1.0, 1.0, 1.0]), lambda t: 0.1, 1000)
x_numer, _ = minimize(f1, cv([1.0, 1.0, 1.0]), lambda t: 0.1, 1000)
print("exact grad  x* =", x_exact.ravel())
print("num  grad   x* =", x_numer.ravel())
print("\nx1 floor is set by float64 cancellation; x2 drifts ~1e-10 over 1000 steps.")

---
## 4. `sgd` — stochastic gradient descent

### The idea
Batch GD evaluates the gradient over **all** $n$ points per step. SGD picks **one point
uniformly at random** and steps on that point's gradient:

$$i_t \sim \mathrm{Uniform}\{0,\dots,n-1\}, \qquad
w^{(t+1)} = w^{(t)} - \eta(t)\, \nabla_w J\!\left(x^{(i_t)}, y^{(i_t)}, w^{(t)}\right)$$

Each step is $n\times$ cheaper and is an *unbiased* estimate of the full gradient, at the
cost of noise — the iterates rattle around the optimum instead of settling on it.

### Contract
`sgd(X, y, J, dJ, w0, step_size_fn, max_iter)` where `X` is `(n, d)`, `y` is `(n, 1)`,
`J(Xi, yi, w)` and `dJ(Xi, yi, w)` take **single-row slices**, and `w0` is `(d, 1)`.

**Returns `w` only** — a `(d, 1)` array, not a tuple. The test compares the return value
directly against a `(d, 1)` array and checks its shape.

### Two details the tests pin down
1. Draw the index with `np.random.randint(n)` — the test seeds NumPy with `0` before each
   case, so the *sequence of draws* must match the reference exactly.
2. Slice with `X[i:i+1, :]` (keeps 2-D shape `(1, d)`), never `X[i]` (collapses to `(d,)`
   and silently breaks the matrix algebra downstream).

Note that `TestSGD` passes your `num_grad` in to build `dJ`, so section 2 must be correct
for this test to pass.

In [ ]:
def sgd(X, y, J, dJ, w0, step_size_fn, max_iter):
    """
    Stochastic gradient descent.

    X            : (n, d) data matrix, one example per ROW
    y            : (n, 1) targets
    J            : (Xi, yi, w) -> scalar   objective on a single example
    dJ           : (Xi, yi, w) -> (d, 1)   gradient on a single example
    w0           : (d, 1) initial weights
    step_size_fn : int -> float
    max_iter     : number of stochastic updates

    Returns w of shape (d, 1).
    """
    n = X.shape[0]
    w = w0
    for t in range(max_iter):
        i = np.random.randint(n)          # uniform sample WITH replacement
        Xi = X[i:i + 1, :]                # (1, d) - the slice keeps it 2-D
        yi = y[i:i + 1, :]                # (1, 1)
        w = w - step_size_fn(t) * dJ(Xi, yi, w)
    return w

In [ ]:
tests.TestSGD(sgd, num_grad)

**Watching the noise.** Below, SGD runs on the same 10-point regression problem the test
uses, recording the *full-batch* objective after every update. The curve drops fast and then
plateaus into a noise band — with a constant step size SGD never converges to a point, it
converges to a *neighbourhood*. Decaying $\eta(t)$ is what shrinks that band.

In [ ]:
import matplotlib.pyplot as plt

X_sgd = np.array([[0.0, 0.1, 0.2, 0.3, 0.42, 0.52, 0.72, 0.78, 0.84, 1.0],
                  [1.0, 1.0, 1.0, 1.0, 1.00, 1.00, 1.00, 1.00, 1.00, 1.0]]).T
y_sgd = np.array([[0.4, 0.6, 1.2, 0.1, 0.22, -0.6, -1.5, -0.5, -0.5, 0.0]]).T


def J_full(X, y, w):
    """Full-batch mean squared error, w includes the offset as its last entry."""
    return float(np.mean((X @ w - y) ** 2))


def dJ_point(Xi, yi, w):
    """Analytic gradient of (Xi w - yi)^2 for a single example."""
    return 2 * Xi.T @ (Xi @ w - yi)


def sgd_trace(X, y, w0, step_size_fn, max_iter, seed=0):
    np.random.seed(seed)
    n = X.shape[0]
    w = w0
    hist = [J_full(X, y, w)]
    for t in range(max_iter):
        i = np.random.randint(n)
        w = w - step_size_fn(t) * dJ_point(X[i:i + 1, :], y[i:i + 1, :], w)
        hist.append(J_full(X, y, w))
    return w, np.array(hist)


w_const, h_const = sgd_trace(X_sgd, y_sgd, cv([0.0, 0.0]), lambda t: 0.1, 1000)
w_decay, h_decay = sgd_trace(X_sgd, y_sgd, cv([0.0, 0.0]), lambda t: 0.5 / (t + 1) ** 0.5, 1000)

plt.figure(figsize=(9, 4))
plt.plot(h_const, lw=0.8, label=r"constant $\eta=0.1$")
plt.plot(h_decay, lw=0.8, label=r"decaying $\eta=0.5/\sqrt{t+1}$")
plt.xlabel("SGD update"); plt.ylabel("full-batch MSE")
plt.title("SGD: constant vs. decaying step size"); plt.yscale("log")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("constant-step w :", w_const.ravel(), " final MSE:", h_const[-1])
print("decaying-step w :", w_decay.ravel(), " final MSE:", h_decay[-1])

---
## 5. `objective_func` — the ridge regression objective

### The maths
Ridge regression = squared loss + an $L_2$ penalty on the weights:

$$J(\theta) \;=\; \underbrace{\frac{1}{n}\sum_{i=1}^{n}\left(x^{(i)\top}\theta - y^{(i)}\right)^2}_{\text{mean squared error}} \;+\; \underbrace{\lambda \lVert \theta \rVert^2}_{\text{regulariser}}$$

In matrix form, with `X` of shape `(n, d)` and `y` of shape `(n, 1)`:

$$J(\theta) = \frac{1}{n}\lVert X\theta - y \rVert^2 + \lambda\,\theta^\top\theta$$

The penalty shrinks coefficients toward zero, trading a little bias for a large drop in
variance — which is what stops the model from chasing noise when features are correlated or
`d` is close to `n`.

### Contract — three things the test checks
1. **Higher-order again**: `objective_func(X, y, lam)` returns a function of `theta`.
   This "closes over" the data so the result can be passed straight to `gd`.
2. **No offset $\theta_0$** in this formulation. Verify by hand on test case 1:
   predictions $= 3 \cdot [5,5,10,16,0] = [15,15,30,48,0]$, residuals $[1,-1,-25,-47,10]$,
   squares sum to $2936$, divided by $n=5$ gives **587.2** ✓. With $\lambda=1$:
   $587.2 + 1\cdot 3^2 =$ **596.2** ✓.
3. **Must return a Python `float`**, not a `(1, 1)` array — the test asserts
   `isinstance(J, float)`. Wrap the result in `float(...)`.

In [ ]:
def objective_func(X, y, lam):
    """
    Build the ridge objective J(theta) = (1/n)||X theta - y||^2 + lam*||theta||^2.

    X   : (n, d)
    y   : (n, 1)
    lam : float, L2 regularisation strength

    Returns f: (d, 1) -> float   (a genuine Python float, as the test asserts)
    """
    n = X.shape[0]

    def f(theta):
        residual = X @ theta - y                      # (n, 1)
        mse_term = float(np.sum(residual ** 2) / n)
        reg_term = float(lam * np.sum(theta ** 2))    # == lam * ||theta||^2
        return mse_term + reg_term

    return f

In [ ]:
tests.TestObjectiveFunc(objective_func)

In [ ]:
# Hand-verification of test case 1, and the return-type requirement.
X_1 = np.array([[5, 5, 10, 16, 0]]).T
Y_1 = np.array([[16, 14, 5, 1, 10]]).T
theta_1 = np.array([[3]])

pred = X_1 @ theta_1
resid = pred - Y_1
print("predictions:", pred.ravel())
print("residuals  :", resid.ravel())
print("MSE        :", float(np.sum(resid ** 2) / 5), "(expected 587.2)")

J = objective_func(X_1, Y_1, 1)(theta_1)
print("J with lam=1:", J, "(expected 596.2)")
print("type(J) is float ->", isinstance(J, float))

---
## 6. `objective_func_grad` — the analytic gradient

### Deriving it
Write $r = X\theta - y$. Then $J(\theta) = \frac{1}{n} r^\top r + \lambda\,\theta^\top\theta$.

Using $\dfrac{\partial}{\partial \theta}\left(r^\top r\right) = 2 X^\top r$ (chain rule, with
$\partial r/\partial\theta = X$) and $\dfrac{\partial}{\partial\theta}\left(\theta^\top\theta\right) = 2\theta$:

$$\boxed{\;\nabla_\theta J \;=\; \frac{2}{n}\,X^\top\!\left(X\theta - y\right) \;+\; 2\lambda\,\theta\;}$$

Shape check: $X^\top$ is `(d, n)`, $(X\theta - y)$ is `(n, 1)`, so the product is `(d, 1)` —
same shape as $\theta$, as every gradient must be.

### Verify on test case 1
$X^\top r = 5(-1) + 5(1) + 10(25) + 16(47) + 0 = 1002$, so
$\frac{2}{5}(1002) = 400.8$ ✓. With $\lambda = 1$: $400.8 + 2(1)(3) = 406.8$ ✓.

In [ ]:
def objective_func_grad(X, y, lam):
    """
    Build the gradient of the ridge objective.

    Returns df: (d, 1) -> (d, 1), computing (2/n) X^T (X theta - y) + 2*lam*theta.
    """
    n = X.shape[0]

    def df(theta):
        residual = X @ theta - y                      # (n, 1)
        return (2 / n) * (X.T @ residual) + 2 * lam * theta   # (d, 1)

    return df

In [ ]:
tests.TestObjectiveFuncGrad(objective_func_grad)

**The single most useful debugging habit in this course:** check an analytic gradient
against `num_grad`. If they disagree beyond $\sim10^{-4}$, the derivation is wrong — and
that bug is otherwise nearly invisible, since a wrong gradient still "trains", just to the
wrong answer.

In [ ]:
rng = np.random.default_rng(42)
X_chk = rng.normal(size=(7, 3))
y_chk = rng.normal(size=(7, 1))
theta_chk = rng.normal(size=(3, 1))

for lam in [0.0, 0.05, 1.0]:
    analytic = objective_func_grad(X_chk, y_chk, lam)(theta_chk)
    numeric = num_grad(objective_func(X_chk, y_chk, lam))(theta_chk)
    print(f"lam={lam:<5} max|analytic - numeric| = {np.max(np.abs(analytic - numeric)):.3e}")

---
## 7. `ridge_gd` — fitting ridge regression by gradient descent

Everything now composes: start from $\theta = \mathbf{0}$, and run `gd` with the objective
from section 5 and the gradient from section 6 for **1000 iterations**.

$$\theta^{(0)} = \mathbf{0}_{d \times 1}, \qquad
\theta^{(t+1)} = \theta^{(t)} - \eta(t)\left[\tfrac{2}{n}X^\top(X\theta^{(t)} - y) + 2\lambda\theta^{(t)}\right]$$

### Contract
`ridge_gd(X, y, lam, step_size_fn)` **returns $\theta$ only** — a `(d, 1)` array, not the
`(x, f)` tuple that `gd` returns. Unpack and discard the objective value.

Two conventions worth flagging:
- **Zero initialisation** is safe here because $J$ is strictly convex (for $\lambda > 0$)
  — a single global minimum, so the starting point only affects how long it takes, not where
  it ends up. That is *not* true of neural networks, where zero init breaks symmetry-breaking.
- **1000 iterations** is what the reference values assume. At $\eta=0.01$ that is more than
  enough to converge: the result below matches the closed-form solution to $\sim10^{-14}$.

In [ ]:
def ridge_gd(X, y, lam, step_size_fn, max_iter=1000):
    """
    Fit ridge regression with gradient descent.

    X            : (n, d)
    y            : (n, 1)
    lam          : L2 regularisation strength
    step_size_fn : int -> float
    max_iter     : number of GD updates (1000 matches the reference values)

    Returns theta of shape (d, 1)  -- NOT the (x, f) tuple that gd returns.
    """
    d = X.shape[1]
    theta_init = np.zeros((d, 1))                     # convex objective -> any init works
    theta, _ = gd(objective_func(X, y, lam),
                  objective_func_grad(X, y, lam),
                  theta_init, step_size_fn, max_iter)
    return theta

In [ ]:
tests.TestRidgeGD(ridge_gd)

---
## 8. Diagnostics — does the iterative solution match the closed form?

Ridge regression has an exact solution,

$$\theta^* = \left(X^\top X + n\lambda I\right)^{-1} X^\top y$$

which `hw03_tests.ridge_analytic` provides. Gradient descent should land on it. Comparing
the two is the strongest possible check that sections 5, 6 and 7 are all consistent: a sign
error or a missing $2/n$ would show up immediately as a mismatch.

In [ ]:
X_train = np.array([
    [2.367446386606739,    8.7520044953452,    8.14485148030185,   6.516389972287224],
    [0.039733614908928905, 3.68480648450503,   5.42213838170197,   6.672864650109663],
    [3.9301662224536074,   0.6049574778006783, 1.7312268364620098, 0.8299188668622759],
]).T
y_train = np.array([[0.31776829626931735, 7.180994658970237,
                     2.402211421436035,   1.300292687028518]]).T

theta_gd = ridge_gd(X_train, y_train, 0.01, lambda t: 0.01)
theta_cf = ridge_analytic(X_train, y_train, 0.01)

print("gradient descent :", theta_gd.ravel())
print("closed form      :", theta_cf.ravel())
print("max abs diff     :", np.max(np.abs(theta_gd - theta_cf)))

### 8.1 Convergence curve

`gd` deliberately returns only the final point, so this helper re-runs the same loop while
recording the objective. Note the straight line on a log scale: gradient descent on a
strongly convex quadratic converges **linearly** (error shrinks by a constant factor each
step), and it flattens out once it hits float64 precision.

In [ ]:
def gd_trace(f, df, x0, step_size_fn, max_iter):
    """Same loop as gd, but records f(x) at every iterate. For diagnostics only."""
    x = x0
    hist = [f(x)]
    for t in range(max_iter):
        x = x - step_size_fn(t) * df(x)
        hist.append(f(x))
    return x, np.array(hist)


lam = 0.01
f_obj = objective_func(X_train, y_train, lam)
df_obj = objective_func_grad(X_train, y_train, lam)

_, hist = gd_trace(f_obj, df_obj, np.zeros((X_train.shape[1], 1)), lambda t: 0.01, 1000)
J_min = f_obj(ridge_analytic(X_train, y_train, lam))

plt.figure(figsize=(9, 4))
plt.semilogy(np.maximum(hist - J_min, 1e-18), lw=1.4)
plt.xlabel("iteration"); plt.ylabel(r"$J(\theta^{(t)}) - J(\theta^*)$")
plt.title(r"Ridge GD convergence ($\eta=0.01$, $\lambda=0.01$)")
plt.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()

print("objective at start :", hist[0])
print("objective at end   :", hist[-1])
print("closed-form optimum:", J_min)

### 8.2 Step size is the whole ball game

Too small and you stall before iteration 1000; too large and the iterates overshoot the
valley floor and oscillate outward to infinity. For a quadratic objective with Hessian
$H = \frac{2}{n}X^\top X + 2\lambda I$, divergence begins at $\eta > 2/\lambda_{\max}(H)$ —
a threshold you can compute exactly here, and only guess at for a neural network.

In [ ]:
H = (2 / X_train.shape[0]) * (X_train.T @ X_train) + 2 * lam * np.eye(X_train.shape[1])
eta_max = 2 / np.max(np.linalg.eigvalsh(H))
print(f"theoretical divergence threshold: eta > {eta_max:.5f}\n")

print(f"{'eta':>9} | {'|theta - theta*|':>18} | status")
print("-" * 48)
with np.errstate(over="ignore", invalid="ignore"):     # divergence is the point here
    for eta in [1e-4, 1e-3, 1e-2, 5e-2, eta_max * 0.99, eta_max * 1.01, 0.3]:
        th = ridge_gd(X_train, y_train, lam, lambda t, e=eta: e)
        err = np.max(np.abs(th - theta_cf))
        if not np.isfinite(err) or err > 1e3:
            status = "DIVERGED"
        elif err < 1e-8:
            status = "converged"
        else:
            status = "too slow, still moving"
        print(f"{eta:>9.5f} | {err:>18.3e} | {status}")

### 8.3 What $\lambda$ actually does

Sweeping the regularisation strength shows the mechanism directly: as $\lambda$ grows, the
fitted coefficients shrink toward zero and the training error rises. The value that
minimises *training* error is always $\lambda = 0$ — which is exactly why $\lambda$ must be
chosen by cross-validation on held-out data, never by fitting error.

In [ ]:
lams = np.logspace(-4, 1, 40)
coefs = np.array([ridge_gd(X_train, y_train, l, lambda t: 0.01, 3000).ravel() for l in lams])
train_mse = np.array([float(np.mean((X_train @ c.reshape(-1, 1) - y_train) ** 2)) for c in coefs])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for j in range(coefs.shape[1]):
    ax[0].semilogx(lams, coefs[:, j], label=fr"$\theta_{j}$")
ax[0].axhline(0, color="k", lw=0.6)
ax[0].set_xlabel(r"$\lambda$"); ax[0].set_ylabel("coefficient")
ax[0].set_title("Coefficient shrinkage path"); ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].semilogx(lams, train_mse, color="crimson")
ax[1].set_xlabel(r"$\lambda$"); ax[1].set_ylabel("training MSE")
ax[1].set_title("Training error rises monotonically with $\\lambda$"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 9. Full regression run

Re-runs every official test in one cell. Use this after any edit — it is the same check the
autograder performs.

In [ ]:
banner = lambda s: print("\n" + "=" * 64 + f"\n  {s}\n" + "=" * 64)

banner("1/7  TestGradientDescent(gd)");            tests.TestGradientDescent(gd)
banner("2/7  TestNumGrad(num_grad)");              tests.TestNumGrad(num_grad)
banner("3/7  TestMinimize(minimize)");             tests.TestMinimize(minimize)
banner("4/7  TestSGD(sgd, num_grad)");             tests.TestSGD(sgd, num_grad)
banner("5/7  TestObjectiveFunc(objective_func)");  tests.TestObjectiveFunc(objective_func)
banner("6/7  TestObjectiveFuncGrad(...)");         tests.TestObjectiveFuncGrad(objective_func_grad)
banner("7/7  TestRidgeGD(ridge_gd)");              tests.TestRidgeGD(ridge_gd)

print("\n" + "=" * 64)
print("  Done - every block above should read 'Passed' / 'All tests passed!'")
print("=" * 64)

---
## 10. Reference card

```python
gd(f, df, x0, step_size_fn, max_iter)        -> (x, f(x))     # exactly max_iter updates
num_grad(f, delta=0.001)                     -> df            # df: (d,1) -> (d,1)
minimize(f, x0, step_size_fn, max_iter)      -> (x, f(x))     # gd + num_grad
sgd(X, y, J, dJ, w0, step_size_fn, max_iter) -> w             # w only, NOT a tuple
objective_func(X, y, lam)                    -> f             # f(theta) -> float
objective_func_grad(X, y, lam)               -> df            # df(theta) -> (d,1)
ridge_gd(X, y, lam, step_size_fn)            -> theta         # theta only, NOT a tuple
```

**The four mistakes that cause almost every failure here**

1. **Off-by-one in the loop.** `gd` must perform `max_iter` updates. Returning after
   `max_iter - 1` gives $x_2 = -98.9$ instead of $-99$ in test 1.
2. **Wrong return arity.** `gd` and `minimize` return a **tuple**; `sgd` and `ridge_gd`
   return a **bare array**. The tests compare these directly and will not unwrap for you.
3. **Shape collapse.** `X[i]` gives `(d,)`; you want `X[i:i+1, :]` → `(1, d)`. NumPy will
   broadcast a 1-D array without complaint and produce silently wrong numbers.
4. **Return type.** `objective_func` must return a Python `float` — `TestObjectiveFunc`
   asserts it explicitly, and a `(1,1)` array fails that assert before any value is compared.

**Ideas worth carrying forward**

- Gradient descent is *local*: on $f(x)=x_0x_1$ (test 2 of section 1) it diverges to
  $-10^8$, because a saddle has no minimum to find.
- A numerical gradient is a debugging tool, not a training tool: $2d$ function evaluations
  per gradient is why backpropagation exists.
- Comparing an analytic gradient to `num_grad` catches derivation errors that otherwise
  show up only as a model that trains to the wrong answer.
- Ridge has a closed form, so GD is unnecessary *here* — but the closed form needs a
  $d \times d$ inverse and exists only for this objective. GD is what generalises.